## Neural networks
This notebook documents Evan's foray into neural networks. I was unable to use Tensorflow and so had to use Pytorch. The highest f1 score I could achieve was 72, and so I believe I am improperly creating my layer structure.

The main strategy was to use brute force, using the 'best overall' functions (SELU and Adam) with a long training time. Ultimately I believe I need to study more about neural networks to achieve performance on par with our other best algorithms

In [27]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import os

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
#metrics
from sklearn.metrics import accuracy_score
from torchmetrics.classification import BinaryF1Score, MulticlassF1Score


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
import torch.optim as optim

import time #used for seeing how long it takes to run programs



In [28]:
#This will state whether neural network is set up to use GPU
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [29]:
train_val = pd.read_csv('data/train_values.csv') #The data values of each building
train_labels = pd.read_csv('data/train_labels.csv') #labels of data as damage severity ('damage_grade')

test_val = pd.read_csv('data/test_values.csv') #data values of contest submition (our AI would have to guess this, 
#but we won't know their 'damage_grade' for sure until DrivenData releases the answers.

#We combine the features and target label for data sampling and other transformations
df = pd.merge(train_val, train_labels, on='building_id')

#with neural networks, we need damage grade to be 0, 1, 2, instead of 1, 2, 3
#I need to remember to correct for this when submitting
df['damage_grade'] = df['damage_grade'] - 1


## Experiments with outlier removal
This code finds all instances that have the same features but a different target outcome (damage grade). They are first removed, then the most common result of the removed duplicates (for each feature combination) are added back in
Over 28 thousand duplicates are operated on, however, removing them did not improve performance. Therefore this code is currently not used.

In [30]:
columnList = df.drop(['damage_grade', 'building_id'], axis = 1).columns.tolist()

duplicates = df[df.duplicated(subset=columnList, keep=False)]
#This code searches for the most common target outcome of the duplicates to add back in to the data
mostCommon = (
    duplicates.groupby(columnList)['damage_grade']
      .agg(lambda x: x.value_counts().idxmax())
      .reset_index()
)

mostCommon.sort_values(by=columnList).head(10)

#remove duplicates
df2 = df.drop_duplicates(subset=columnList, keep=False).copy()

#add back the most common without adding back the outliers
df2 = pd.concat([df2, mostCommon]) 
df2 = df2.fillna(0)

In [31]:
#Throughout testing we would sample the data so test runs don't take so long
#Currently we have changed it to a full sample to to see what model best utilises the full dataset
dataSample = df.sample(frac=1)

#Default Features to drop:
#building_id: for recordkeeping rather than data
#The target variable: damage_grade
features = dataSample.drop(['building_id', 'damage_grade'], axis = 1)

#These tables are in string type and must be converted
categoryTables = ['land_surface_condition', 'foundation_type', 'roof_type', 'ground_floor_type', 'other_floor_type', 'position', 'plan_configuration', 'legal_ownership_status']

# print(categoryTables.shape)
one_hot_tables = pd.get_dummies(features[categoryTables], dtype=int, drop_first=True)




#Now we replace our string tables with the encoded tables
X = features.drop(categoryTables, axis = 1)
X = pd.concat([X, one_hot_tables], axis = 1)
print('data shape:',X.shape)
X.head(3)

data shape: (260601, 60)


,geo_level_1_id,geo_level_2_id,geo_level_3_id,count_floors_pre_eq,age,area_percentage,height_percentage,has_superstructure_adobe_mud,has_superstructure_mud_mortar_stone,has_superstructure_stone_flag,has_superstructure_cement_mortar_stone,has_superstructure_mud_mortar_brick,has_superstructure_cement_mortar_brick,has_superstructure_timber,has_superstructure_bamboo,has_superstructure_rc_non_engineered,has_superstructure_rc_engineered,has_superstructure_other,count_families,has_secondary_use,has_secondary_use_agriculture,has_secondary_use_hotel,has_secondary_use_rental,has_secondary_use_institution,has_secondary_use_school,has_secondary_use_industry,has_secondary_use_health_post,has_secondary_use_gov_office,has_secondary_use_use_police,has_secondary_use_other,land_surface_condition_o,land_surface_condition_t,foundation_type_i,foundation_type_r,foundation_type_u,foundation_type_w,roof_type_q,roof_type_x,ground_floor_type_m,ground_floor_type_v,ground_floor_type_x,ground_floor_type_z,other_floor_type_q,other_floor_type_s,other_floor_type_x,position_o,position_s,position_t,plan_configuration_c,plan_configuration_d,plan_configuration_f,plan_configuration_m,plan_configuration_n,plan_configuration_o,plan_configuration_q,plan_configuration_s,plan_configuration_u,legal_ownership_status_r,legal_ownership_status_v,legal_ownership_status_w
34831,20,158,4613,2,5,8,7,0,0,0,0,0,1,1,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0
143085,7,430,12378,2,35,8,3,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0
20595,26,1132,3071,2,10,6,7,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0


In [32]:
# scaler = MinMaxScaler()
scaler = StandardScaler()
X = scaler.fit_transform(X)
print('X:',X.shape)

y = dataSample['damage_grade']
print('y:',y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=44, stratify=y)
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X: (260601, 60)
y: (260601,)


In [33]:


X_train_tensor = torch.tensor(X_train)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)

X_test_tensor = torch.tensor(X_test)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

In [34]:
print(X_train_tensor)
print(X_train_tensor.size())

tensor([[-0.1121, -1.1632,  0.3461,  ..., -0.0754,  0.1962, -0.1019],
        [ 0.8837,  0.0628, -0.2586,  ..., -0.0754,  0.1962, -0.1019],
        [ 0.3858,  1.6475, -1.5840,  ..., -0.0754,  0.1962, -0.1019],
        ...,
        [ 0.8837, -1.4394, -0.6003,  ..., -0.0754,  0.1962, -0.1019],
        [ 0.2614, -0.8507,  0.7430,  ..., -0.0754,  0.1962, -0.1019],
        [ 0.8837, -0.9403,  1.0301,  ..., -0.0754,  0.1962, -0.1019]],
       dtype=torch.float64)
torch.Size([208480, 60])


In [41]:

#This code was adapted from the documentation. It is used to recombine data to feed into the data loaders
#I am not entirely sure why this is needed but I believe it is caused by the specific structure of tensors
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TabularDataset(X_train, y_train.to_numpy())
test_dataset = TabularDataset(X_test, y_test.to_numpy())

#Dataloaders are iterables over the dataset. When iterating over it, 
#it will load in random order from the dataset (including the data-sample and the target/label)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [36]:
model = nn.Sequential(

    nn.Linear(60, 70),
    nn.SELU(),
    nn.Linear(70, 60),
    nn.SELU(),
    nn.Linear(60, 50),
    nn.SELU(),
    nn.Linear(50, 40),
    nn.SELU(),
    nn.Linear(40, 25),
    nn.SELU(),
    nn.Linear(25, 20),
    nn.SELU(),
    nn.Linear(20, 20),
    nn.SELU(),
    nn.Linear(20, 10),
    nn.SELU(),
    nn.Linear(10, 3)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training loop
Ever 20 instances we observe model accuracy
For a much shorter training cycle, you can set num_epochs

In [37]:
num_epochs = 800
totalInstancesTrained = 0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch_x, batch_y in train_loader:
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        totalInstancesTrained += batch_size

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}")

    #every 20 epochs test model
    ###########################
    if epoch % 20 == 0:
        model.eval()
        correct = 0
        total = 0
        #this loop gets the training accuracy
        with torch.no_grad():
            for batch_x, batch_y in train_loader:
                outputs = model(batch_x)
                _, predicted = torch.max(outputs, 1)
        
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        accuracy = correct / total
        print(f"Train Accuracy: {accuracy:.4f}")
        #this loop gets the testing accuracy
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                outputs = model(batch_x)
                _, predicted = torch.max(outputs, 1)
        
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        
        accuracy = correct / total
        print(f"Test Accuracy: {accuracy:.4f}")
        #testing f1 score
        metric = MulticlassF1Score(num_classes=3, average="micro")
        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                preds = torch.argmax(outputs, dim=1)
                metric.update(preds, labels)
        
        f1 = metric.compute()
        print(f"Test F1 Score: {f1:.4f}")

Epoch 1/800, Loss: 5045.6611
Train Accuracy: 0.6358
Test Accuracy: 0.6354
Test F1 Score: 0.6339
Epoch 2/800, Loss: 4793.6868
Epoch 3/800, Loss: 4719.1760
Epoch 4/800, Loss: 4677.9018
Epoch 5/800, Loss: 4633.7711
Epoch 6/800, Loss: 4595.3428
Epoch 7/800, Loss: 4567.1422
Epoch 8/800, Loss: 4535.6836
Epoch 9/800, Loss: 4511.7798
Epoch 10/800, Loss: 4492.5195
Epoch 11/800, Loss: 4470.9668
Epoch 12/800, Loss: 4453.5814
Epoch 13/800, Loss: 4430.4495
Epoch 14/800, Loss: 4419.7015
Epoch 15/800, Loss: 4403.6989
Epoch 16/800, Loss: 4386.9998
Epoch 17/800, Loss: 4374.6681
Epoch 18/800, Loss: 4362.3148
Epoch 19/800, Loss: 4349.8682
Epoch 20/800, Loss: 4335.2722
Epoch 21/800, Loss: 4326.0631
Train Accuracy: 0.6906
Test Accuracy: 0.6882
Test F1 Score: 0.6789
Epoch 22/800, Loss: 4318.7509
Epoch 23/800, Loss: 4309.5389
Epoch 24/800, Loss: 4302.4075
Epoch 25/800, Loss: 4291.0592
Epoch 26/800, Loss: 4285.1650
Epoch 27/800, Loss: 4273.5334
Epoch 28/800, Loss: 4266.8058
Epoch 29/800, Loss: 4259.8235
Epoch

## Final testing loop

In [38]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch_x, batch_y in train_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = correct / total
print(f"Train Accuracy: {accuracy:.4f}")

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, predicted = torch.max(outputs, 1)

        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

Train Accuracy: 0.7452
Test Accuracy: 0.7321


In [39]:
print(totalInstancesTrained)

166784000
